# A5 — Brushstroke Geometric Features

**Goal:** Extract individual brushstroke paths and compute their geometric statistics. Visualise length, width, curvature, and orientation distributions for Manet vs a contemporary.

**Key paper:** Li, Yao, Hendriks & Wang (2012), *IEEE T-PAMI* 34(6):1159–1176 — rhythmic brushstrokes, van Gogh

**Key question for Manet:** Can individual brushstrokes be reliably extracted from Manet's flat-colour, contour-dominated style? Or does the flat-area segmentation break the method?

---

### Mathematics

**Extraction pipeline:**
1. Canny edge detection → binary edge map
2. Morphological skeletonisation → 1-pixel-wide paths
3. Connected component labelling → individual brushstroke regions
4. Per-brushstroke feature extraction

**Per-brushstroke descriptors:**
- **Length** $L$: arc length along skeletonised path
- **Width** $W$: mean perpendicular cross-section width
- **Aspect ratio** $AR = L / W$
- **Curvature** $\kappa = \frac{|y''|}{(1 + y'^2)^{3/2}}$ (mean absolute curvature)
- **Orientation** $\theta$: dominant angle from structure tensor

**Expected for Manet:** Long $L$, low $\kappa$ (flat colour areas with strong confident contours)  
**Expected for da Vinci (sfumato):** Short $L$, high $\kappa$, dense packing

In [ ]:
import sys
sys.path.insert(0, '../..')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from notebooks.research.utils import load_image, to_gray, show_pair
from skimage.feature import canny
from skimage.morphology import skeletonize, label, remove_small_objects
from skimage.measure import regionprops
from scipy.ndimage import gaussian_filter

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
PATH_MANET        = Path('../../data/manet/manet_example.tif')
PATH_CONTEMPORARY = Path('../../data/contemporary/contemporary_example.tif')
CANNY_SIGMA    = 2.0    # Gaussian smoothing before edge detection
CANNY_LOW      = 0.05   # Low hysteresis threshold
CANNY_HIGH     = 0.15   # High hysteresis threshold
MIN_STROKE_LEN = 10     # Minimum brushstroke length in pixels
# ─────────────────────────────────────────────────────────────────────────────

img_m  = load_image(PATH_MANET)
img_c  = load_image(PATH_CONTEMPORARY)
gray_m = to_gray(img_m)
gray_c = to_gray(img_c)
show_pair(img_m, img_c)

---
## 1. Edge Detection → Skeletonisation → Brushstroke Regions

In [ ]:
def extract_strokes(gray: np.ndarray,
                    sigma: float = 2.0,
                    low: float = 0.05, high: float = 0.15,
                    min_len: int = 10) -> tuple:
    """
    Returns (edge_map, skeleton, labelled_skeleton, n_strokes).
    """
    # Step 1: Canny edges
    edges = canny(gray, sigma=sigma, low_threshold=low, high_threshold=high)

    # Step 2: Skeletonise — thin edges to 1-pixel paths
    skeleton = skeletonize(edges)

    # Step 3: Remove very short strokes (noise)
    labelled = label(skeleton)
    cleaned  = remove_small_objects(labelled, min_size=min_len)
    labelled_clean = label(cleaned > 0)
    n = labelled_clean.max()

    return edges, skeleton, labelled_clean, n


edges_m, skel_m, lab_m, n_m = extract_strokes(
    gray_m, CANNY_SIGMA, CANNY_LOW, CANNY_HIGH, MIN_STROKE_LEN)
edges_c, skel_c, lab_c, n_c = extract_strokes(
    gray_c, CANNY_SIGMA, CANNY_LOW, CANNY_HIGH, MIN_STROKE_LEN)

print(f'Manet:        {n_m} brushstroke segments extracted')
print(f'Contemporary: {n_c} brushstroke segments extracted')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Brushstroke extraction pipeline', fontsize=12)

for row, (name, img, edges, skel, lab) in enumerate([
    ('Manet',        img_m, edges_m, skel_m, lab_m),
    ('Contemporary', img_c, edges_c, skel_c, lab_c),
]):
    axes[row, 0].imshow(img)
    axes[row, 0].set_title(f'{name} — original')
    axes[row, 0].axis('off')

    axes[row, 1].imshow(edges, cmap='gray')
    axes[row, 1].set_title(f'Canny edges (σ={CANNY_SIGMA})')
    axes[row, 1].axis('off')

    # Overlay skeleton on original
    overlay = img.copy()
    overlay[skel > 0] = [1.0, 0.2, 0.2]   # Red skeleton
    axes[row, 2].imshow(overlay)
    axes[row, 2].set_title(f'Skeleton overlay ({lab.max()} strokes)')
    axes[row, 2].axis('off')

plt.tight_layout()
plt.show()

---
## 2. Per-Brushstroke Feature Extraction

In [ ]:
def compute_curvature(coords: np.ndarray) -> float:
    """
    Mean absolute curvature κ = |y''| / (1 + y'²)^{3/2}
    along a sequence of (row, col) coordinates.
    Returns 0 if fewer than 3 points.
    """
    if len(coords) < 3:
        return 0.0
    y, x = coords[:, 0].astype(float), coords[:, 1].astype(float)
    # Parameterise by arc length
    dx = np.gradient(x)
    dy = np.gradient(y)
    d2x = np.gradient(dx)
    d2y = np.gradient(dy)
    denom = (dx**2 + dy**2)**1.5 + 1e-9
    kappa = np.abs(dx * d2y - dy * d2x) / denom
    return float(kappa.mean())


def extract_stroke_features(labelled: np.ndarray) -> dict:
    """
    Extract geometric features from all brushstroke regions.
    Returns dict of lists: L, W, AR, kappa, theta.
    """
    props_list = regionprops(labelled)
    L_list, W_list, AR_list, kappa_list, theta_list = [], [], [], [], []

    for prop in props_list:
        # Arc length ≈ number of pixels in skeletonised region
        L = prop.area
        # Width from bounding box minor axis
        W = max(prop.minor_axis_length, 1.0)
        AR = L / W

        # Orientation from region's major axis (in radians)
        theta = prop.orientation  # [-π/2, π/2]

        # Curvature: get pixel coordinates of the region
        coords = prop.coords  # (n_pixels, 2)
        # Sort by arc order (approximate: sort by position along major axis)
        proj_axis = coords[:, 0] * np.cos(theta) + coords[:, 1] * np.sin(theta)
        sort_idx  = np.argsort(proj_axis)
        kappa = compute_curvature(coords[sort_idx])

        L_list.append(L)
        W_list.append(W)
        AR_list.append(AR)
        kappa_list.append(kappa)
        theta_list.append(theta)

    return {'L': np.array(L_list), 'W': np.array(W_list), 'AR': np.array(AR_list),
            'kappa': np.array(kappa_list), 'theta': np.array(theta_list)}


strokes_m = extract_stroke_features(lab_m)
strokes_c = extract_stroke_features(lab_c)
print(f'Manet:        {len(strokes_m["L"])} strokes  |  mean L={strokes_m["L"].mean():.1f}px  mean κ={strokes_m["kappa"].mean():.4f}')
print(f'Contemporary: {len(strokes_c["L"])} strokes  |  mean L={strokes_c["L"].mean():.1f}px  mean κ={strokes_c["kappa"].mean():.4f}')

---
## 3. Distribution Plots — L, W, AR, κ, θ

In [ ]:
features_to_plot = [
    ('L',     'Stroke length (px)',            (0, np.percentile(np.concatenate([strokes_m['L'], strokes_c['L']]), 98))),
    ('W',     'Stroke width (px)',             (0, np.percentile(np.concatenate([strokes_m['W'], strokes_c['W']]), 98))),
    ('AR',    'Aspect ratio L/W',             (0, np.percentile(np.concatenate([strokes_m['AR'], strokes_c['AR']]), 98))),
    ('kappa', 'Mean curvature κ',             (0, np.percentile(np.concatenate([strokes_m['kappa'], strokes_c['kappa']]), 98))),
    ('theta', 'Orientation θ (rad)',          (-np.pi/2, np.pi/2)),
]

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
fig.suptitle('Brushstroke feature distributions: Manet (blue) vs Contemporary (orange)\n'
             'Hypothesis: Manet = longer L, lower κ', fontsize=11)

for ax, (feat, xlabel, xlim) in zip(axes, features_to_plot):
    data_m = strokes_m[feat]
    data_c = strokes_c[feat]

    # Clip to xlim for display
    data_m_c = np.clip(data_m, xlim[0], xlim[1])
    data_c_c = np.clip(data_c, xlim[0], xlim[1])

    ax.hist(data_m_c, bins=40, density=True, alpha=0.6, color='#2196F3',
            label=f'Manet\nμ={data_m.mean():.2f}')
    ax.hist(data_c_c, bins=40, density=True, alpha=0.6, color='#FF5722',
            label=f'Contemp.\nμ={data_c.mean():.2f}')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Density')
    ax.set_xlim(xlim)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## 4. Curvature Heatmap

In [ ]:
def curvature_heatmap(labelled: np.ndarray, stroke_feats: dict) -> np.ndarray:
    """Paint κ value of each stroke region onto a spatial map."""
    heat = np.zeros(labelled.shape)
    props = regionprops(labelled)
    for i, prop in enumerate(props):
        if i < len(stroke_feats['kappa']):
            coords = prop.coords
            heat[coords[:, 0], coords[:, 1]] = stroke_feats['kappa'][i]
    return heat


heat_m = curvature_heatmap(lab_m, strokes_m)
heat_c = curvature_heatmap(lab_c, strokes_c)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Curvature κ heatmap — bright = high curvature (complex stroke path)\n'
             'Manet hypothesis: high κ at contours, low κ in flat areas', fontsize=11)

vmax = np.percentile(np.concatenate([heat_m[heat_m > 0], heat_c[heat_c > 0]]), 95) if (heat_m > 0).any() else 1

for col, (name, img, heat) in enumerate([
    ('Manet', img_m, heat_m), ('Contemporary', img_c, heat_c)
]):
    axes[0, col].imshow(img)
    axes[0, col].set_title(f'{name}')
    axes[0, col].axis('off')

    im = axes[1, col].imshow(heat, cmap='plasma', vmin=0, vmax=vmax)
    plt.colorbar(im, ax=axes[1, col], fraction=0.03, label='κ')
    axes[1, col].set_title(f'{name} — curvature map')
    axes[1, col].axis('off')

plt.tight_layout()
plt.show()

---
## 5. Summary Feature Vector

In [ ]:
def stroke_feature_vector(stroke_feats: dict) -> dict:
    """Statistical moments of brushstroke distributions → ~20 features."""
    result = {}
    for key in ['L', 'W', 'AR', 'kappa', 'theta']:
        data = stroke_feats[key]
        result[f'{key}_mean'] = data.mean()
        result[f'{key}_std']  = data.std()
        result[f'{key}_skew'] = float(np.mean(((data - data.mean()) / (data.std() + 1e-9))**3))
        result[f'{key}_kurt'] = float(np.mean(((data - data.mean()) / (data.std() + 1e-9))**4))
    result['n_strokes'] = len(stroke_feats['L'])
    return result


import pandas as pd
fv_m = stroke_feature_vector(strokes_m)
fv_c = stroke_feature_vector(strokes_c)

df = pd.DataFrame({'Manet': fv_m, 'Contemporary': fv_c})
df['|Difference|'] = (df['Manet'] - df['Contemporary']).abs()
df['Rel. diff (%)'] = (df['|Difference|'] / (df['Manet'].abs() + 1e-9) * 100).round(1)
print(df.sort_values('|Difference|', ascending=False).round(4).to_string())

---
## 6. Key Observations

| Question | Observation |
|---|---|
| Were brushstrokes reliably extracted, or is segmentation noisy? | |
| Does Manet show longer L than the contemporary? | |
| Does κ (curvature) differentiate the two? | |
| Which moment (mean/std/skewness/kurtosis) of which feature differs most? | |
| Would flat-colour areas cause under-extraction of brushstrokes? | |
| Is this method viable for Manet, or better suited for da Vinci/van Gogh? | |